# Workstream B: Learning Dynamics and Trainability



## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Output folder

In [2]:
import os, glob, re, json, csv, math, shutil
from collections import Counter, defaultdict
import numpy as np

MYDRIVE = "/content/drive/MyDrive"

ROOT = None
for cand in sorted(glob.glob(os.path.join(MYDRIVE, "**", "WISER Results"), recursive=True)):
    if os.path.isdir(cand):
        ROOT = cand; break
if ROOT is None:
    ROOT = os.path.join(MYDRIVE, "WISER Results"); os.makedirs(ROOT, exist_ok=True)
    print(f"[note] created {ROOT}")
OUT = os.path.join(ROOT, "Phase 3", "WSB")
os.makedirs(OUT, exist_ok=True)
print("WISER Results root :", ROOT)
print("output folder      :", OUT)

HIST, GRAD, RES, GEN = {}, [], {}, []
ACCOUNT = []        # one entry per checkpoint file: kept or excluded, with a reason
cov = defaultdict(list)

WISER Results root : /content/drive/MyDrive/WISER Results
output folder      : /content/drive/MyDrive/WISER Results/Phase 3/WSB


## 3. Inventory

In [3]:
print("scanning MyDrive ...\n")
INV = {
    "checkpoints":    glob.glob(os.path.join(MYDRIVE, "**", "checkpoints", "*.pt"), recursive=True),
    "gradients":      glob.glob(os.path.join(MYDRIVE, "**", "*gradient_variance.npz"), recursive=True),
    "generalization": glob.glob(os.path.join(MYDRIVE, "**", "generalization", "*.npz"), recursive=True),
    "master_csv":     glob.glob(os.path.join(MYDRIVE, "**", "master_results.csv"), recursive=True),
    "landscape":      glob.glob(os.path.join(MYDRIVE, "**", "*loss_landscape*"), recursive=True),
}
for k, v in INV.items(): print(f"{k:16s}: {len(v):5d} files")

print("\n--- where they live ---")
for k, v in INV.items():
    if not v: continue
    print(f"\n[{k}]")
    for folder, cnt in Counter(os.path.dirname(p) for p in v).most_common(15):
        ex = os.path.basename(next(q for q in v if os.path.dirname(q) == folder))
        print(f"  {cnt:5d}  {folder.replace(MYDRIVE,'').lstrip('/')}")
        print(f"         e.g. {ex}")
N_CKPT = len(INV["checkpoints"])

scanning MyDrive ...

checkpoints     :   283 files
gradients       :     6 files
generalization  :     6 files
master_csv      :    10 files
landscape       :     2 files

--- where they live ---

[checkpoints]
     88  WISER Results/Phase 2/Djabon Phase 2/Additionnal computations/GAAF-PINN/qapinn_runs_gaaf/checkpoints
         e.g. MS_burgers_n3_gaaf_s1234.pt
     40  WISER Results/Phase 2/Djabon Phase 2/Tiers12_multi_seed/qapinn_runs/checkpoints
         e.g. MS_burgers_n3_qapinn_s1234.pt
     37  WISER Results/Phase 2/Djabon Phase 2/Additionnal computations/Heat Multi_seed_q38/qapinn_runs_heat/checkpoints
         e.g. MS_heat_n3_qapinn_s1234.pt
     36  WISER Results/Phase 2/Djabon Phase 2/Additionnal computations/Shah & al Reproduction/qapinn_runs_shah/checkpoints
         e.g. REPRO_shah_easy_n3_shah_s1234.pt
     22  WISER Results/Phase 2 Hard Burger/GAAF-PINN/qapinn_runs_hardnu_gaaf/checkpoints
         e.g. MS_hard_burgers_n3_gaaf_s1234.pt
     18  WISER Results/Phase 2 Hard 

## 4. Tag parsing

Every naming convention used anywhere in the project is handled. Anything that still fails to
parse is listed by name in the accounting table below, never dropped in silence.

In [4]:
RE_MS    = re.compile(r"MS_(?P<hard>hard_)?(?P<pde>burgers|heat)_n(?P<n>\d+)_(?P<model>qapinn|twin|gaaf)_s(?P<seed>\d+)(?P<retry>_r\d+)?")
RE_V5    = re.compile(r"V5_(?P<pde>burgers|heat)_q(?P<n>\d+)")
RE_T1    = re.compile(r"T1_(?P<pde>burgers|heat)_q(?P<n>\d+)")
RE_CT    = re.compile(r"CT_(?P<pde>burgers|heat)_n(?P<n>\d+)")
RE_T2    = re.compile(r"T2_(?P<kind>hero|smoke)_q(?P<n>\d+)")
RE_B     = re.compile(r"^B(?P<idx>\d+)_q(?P<n>\d+)_(?P<ent>\w+?)_(?P<meas>expval|probs)")
RE_REPRO = re.compile(r"REPRO_shah")

def parse_tag(tag):
    m = RE_MS.search(tag)
    if m:
        d = m.groupdict()
        return dict(pde=d["pde"], n_feat=int(d["n"]), model=d["model"], seed=int(d["seed"]),
                    nu=("0.01/pi" if d["hard"] else "0.05"), retry=bool(d["retry"]),
                    family="MS", note="")
    m = RE_V5.search(tag)
    if m:
        return dict(pde=m.group("pde"), n_feat=int(m.group("n")), model="qapinn", seed=1234,
                    nu="0.05", retry=False, family="V5", note="single-run sweep, global SEED")
    m = RE_T1.search(tag)
    if m:
        return dict(pde=m.group("pde"), n_feat=int(m.group("n")), model="qapinn", seed=1234,
                    nu="0.05", retry=False, family="T1", note="Tier-1 early run")
    m = RE_CT.search(tag)
    if m:
        return dict(pde=m.group("pde"), n_feat=int(m.group("n")), model="twin", seed=1234,
                    nu="0.05", retry=False, family="CT", note="Tier-1/2 classical twin")
    m = RE_T2.search(tag)
    if m:
        return dict(pde="burgers", n_feat=int(m.group("n")), model="qapinn", seed=1234,
                    nu="0.01/pi", retry=False, family="T2",
                    note=f"Tier-2 {m.group('kind')} run")
    m = RE_B.search(tag)
    if m:
        return dict(pde="burgers", n_feat=int(m.group("n")), model="qapinn", seed=1234,
                    nu="0.05", retry=False, family="ablation",
                    note=f"Track-B ablation: ent={m.group('ent')}, meas={m.group('meas')}")
    if RE_REPRO.search(tag):
        return dict(pde="burgers", n_feat=None, model=None, seed=None, nu="0.05",
                    retry=False, family="repro", note="Shah reproduction, out of WS-B scope")
    return None

probe = Counter()
for p in INV["checkpoints"]:
    m = parse_tag(os.path.basename(p)[:-3])
    probe[m["family"] if m else "UNPARSED"] += 1
print("checkpoint tags by family:", dict(probe))

checkpoint tags by family: {'UNPARSED': 4, 'ablation': 12, 'CT': 6, 'V5': 12, 'MS': 205, 'T1': 6, 'T2': 2, 'repro': 36}


## 5. Load histories, with every file accounted for

The assertion at the end is the safeguard: kept + excluded must equal the number of checkpoint
files found. If it fails, something is being lost silently and the run stops.

In [ ]:
import torch

DEAD_RATIO = 0.5   # a run whose loss never falls below half its starting value is flagged

seen_tags = {}
for p in sorted(INV["checkpoints"]):
    tag = os.path.basename(p)[:-3]
    rel = p.replace(MYDRIVE, "").lstrip("/")
    meta = parse_tag(tag)

    if meta is None:
        ACCOUNT.append(dict(tag=tag, path=rel, decision="excluded",
                            reason="tag did not parse")); continue
    if meta["family"] == "repro":
        ACCOUNT.append(dict(tag=tag, path=rel, decision="excluded",
                            reason="Shah reproduction, outside WS-B scope")); continue
    if tag in seen_tags:
        ACCOUNT.append(dict(tag=tag, path=rel, decision="excluded",
                            reason=f"same tag already loaded from {seen_tags[tag]}")); continue
    try:
        ck = torch.load(p, map_location="cpu", weights_only=False)
    except Exception as e:
        ACCOUNT.append(dict(tag=tag, path=rel, decision="excluded",
                            reason=f"unreadable: {type(e).__name__}")); continue
    h = ck.get("history")
    if not isinstance(h, dict) or not h.get("loss"):
        ACCOUNT.append(dict(tag=tag, path=rel, decision="excluded",
                            reason="checkpoint holds no loss history")); continue

    y = np.asarray(h["loss"], float)
    dead = bool(np.min(y) > DEAD_RATIO * y[0]) if len(y) > 1 else True
    seen_tags[tag] = rel
    HIST[tag] = dict(tag=tag, path=rel, meta=meta,
                     loss=y,
                     mse_u=np.asarray(h.get("mse_u", []), float),
                     mse_f=np.asarray(h.get("mse_f", []), float),
                     phase=np.asarray(h.get("phase", []), float),
                     wall_s=float(h.get("wall_s", float("nan"))),
                     dead=dead)
    ACCOUNT.append(dict(tag=tag, path=rel, decision="kept",
                        reason="dead run (loss never fell materially)" if dead else ""))

kept = [a for a in ACCOUNT if a["decision"] == "kept"]
excl = [a for a in ACCOUNT if a["decision"] == "excluded"]
print(f"checkpoint files found : {N_CKPT}")
print(f"  kept (histories)     : {len(kept)}")
print(f"  excluded             : {len(excl)}")
print("\nexclusions by reason:")
for r, c in Counter(a["reason"] for a in excl).most_common():
    print(f"  {c:4d}  {r}")
    for a in [x for x in excl if x["reason"] == r][:4]:
        print(f"          e.g. {a['tag']}")

assert len(kept) + len(excl) == N_CKPT, (
    f"ACCOUNTING FAILURE: {len(kept)}+{len(excl)} != {N_CKPT}. Files are being lost silently.")
print(f"\n[OK] accounting closes: {len(kept)} + {len(excl)} = {N_CKPT}")

dead_runs = [t for t, v in HIST.items() if v["dead"]]
print(f"\ndead runs detected from their own histories: {len(dead_runs)}")
for t in dead_runs: print("   ", t, f"(loss {HIST[t]['loss'][0]:.3e} -> min {HIST[t]['loss'].min():.3e})")

print("\n--- coverage (excluding dead runs) ---")
for t, v in HIST.items():
    if v["dead"]: continue
    m = v["meta"]
    cov[(m["nu"], m["pde"], m["model"], m["n_feat"], m["family"])].append(
        f"{m['seed']}r" if m["retry"] else str(m["seed"]))
for k in sorted(cov, key=lambda z: (z[0], z[1], z[2] or "", z[3] if z[3] else 0, z[4])):
    print(f"  nu={k[0]:8s} {k[1]:8s} {str(k[2]):7s} n={str(k[3]):>4} [{k[4]:9s}]: {sorted(cov[k])}")
print("\n('1234r' = retry, kept separately. Family tags show which study a run came from.)")

## 6. Gradient variance, with duplicate detection 

In [ ]:
SERIES = {}
grad_fail = []
for p in sorted(INV["gradients"]):
    rel = p.replace(MYDRIVE, "").lstrip("/")
    run = os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(p))))
    try:
        d = np.load(p, allow_pickle=True)
    except Exception as e:
        grad_fail.append((rel, type(e).__name__)); continue
    keys = list(d.files)
    xkey = "n_feat" if "n_feat" in keys else ("qubits" if "qubits" in keys else None)
    if xkey is None:
        grad_fail.append((rel, f"no qubit axis; keys={keys}")); continue
    xs = np.asarray(d[xkey]).ravel()
    for k in keys:
        if k == xkey: continue
        ys = np.asarray(d[k]).ravel()
        if ys.shape != xs.shape: continue
        model = ("twin" if "twin" in k else "gaaf" if "gaaf" in k else "qapinn")
        nu = "0.01/pi" if "hard" in rel.lower() else "0.05"
        SERIES[(rel, k)] = dict(path=rel, run=run, series=k, model=model, nu=nu,
                                n=[int(a) for a in xs], var=[float(b) for b in ys],
                                labelled=(k != "var"))

print(f"series found: {len(SERIES)}   files failed: {len(grad_fail)}")
for a, b in grad_fail: print(f"   [skip] {a}: {b}")

# --- numerical duplicate detection ------------------------------------------
dupes = {}
ks = sorted(SERIES)
for i in range(len(ks)):
    for j in range(i+1, len(ks)):
        A, B = SERIES[ks[i]], SERIES[ks[j]]
        if A["model"] != B["model"] or A["nu"] != B["nu"]: continue
        a, b = dict(zip(A["n"], A["var"])), dict(zip(B["n"], B["var"]))
        common = sorted(set(a) & set(b))
        if len(common) < 2: continue
        if all(abs(a[n]-b[n]) < 1e-12 for n in common):
            longer, shorter = (ks[i], ks[j]) if len(A["n"]) >= len(B["n"]) else (ks[j], ks[i])
            dupes[shorter] = longer

print(f"\nexact-duplicate series detected: {len(dupes)}")
for s, l in dupes.items():
    print(f"   DROP {s[0]} :: {s[1]}")
    print(f"   because it is identical (16 s.f.) to")
    print(f"        {l[0]} :: {l[1]}")

GRAD = []
for k, S in SERIES.items():
    if k in dupes: continue
    for n, v in zip(S["n"], S["var"]):
        GRAD.append(dict(path=S["path"], run=S["run"], series=S["series"], model=S["model"],
                         nu=S["nu"], n_feat=n, variance=v, model_labelled=S["labelled"]))
print(f"\nunique gradient rows kept: {len(GRAD)}")

print("\n--- every unique series ---")
for k in sorted(SERIES):
    if k in dupes: continue
    S = SERIES[k]
    flag = "" if S["labelled"] else "   [model NOT recorded in file; attributed to QAPINN]"
    print(f"\n  {S['path']}")
    print(f"     series={S['series']:11s} model={S['model']:7s} nu={S['nu']:8s}{flag}")
    print(f"        n   = {S['n']}")
    print(f"        var = {['%.4g' % v for v in S['var']]}")

# --- magnitude audit --------------------------------------------------------
print("\n--- magnitude audit (are any series on a different scale?) ---")
for nu in sorted({g["nu"] for g in GRAD}):
    med = {m: np.median([g["variance"] for g in GRAD if g["nu"] == nu and g["model"] == m])
           for m in sorted({g["model"] for g in GRAD if g["nu"] == nu})}
    print(f"  nu={nu}: median variance by model = "
          + ", ".join(f"{m}={v:.4g}" for m, v in med.items()))
    if len(med) > 1:
        hi, lo = max(med.values()), min(med.values())
        if hi / lo > 100:
            big = [m for m, v in med.items() if v == hi][0]
            print(f"     [FLAG] '{big}' is {hi/lo:.0f}x the smallest. Do NOT read this as a")
            print( "            physical effect until the cause is established: the pooled")
            print( "            first-layer variance may be dominated by one parameter with a")
            print( "            different scale (e.g. GAAF's adaptive slope).")
missing = [(nu, m) for nu in ["0.05", "0.01/pi"] for m in ["qapinn", "twin", "gaaf"]
           if not any(g["nu"] == nu and g["model"] == m for g in GRAD)]
if missing:
    print("\n[MISSING] no gradient-variance data for:")
    for nu, m in missing: print(f"     nu={nu}  model={m}")

## 7. Final results and generalization 

In [ ]:
def detect_kind(row):
    meas = (row.get("measurement") or "").strip().lower()
    ent  = (row.get("entanglement") or "").strip().lower()
    if meas == "expval": return "qapinn"
    if meas == "classical" or ent == "classical": return "twin"
    if meas == "gaaf" or ent == "gaaf": return "gaaf"
    tag = (row.get("tag") or "").lower()
    for k in ("qapinn", "twin", "gaaf"):
        if k in tag: return k
    return None

res_unresolved = []
for p in sorted(INV["master_csv"]):
    try: rows = list(csv.DictReader(open(p)))
    except Exception as e:
        res_unresolved.append((p, str(e))); continue
    for row in rows:
        tag = row.get("tag", "")
        meta, kind = parse_tag(tag), detect_kind(row)
        if meta is None or kind is None:
            res_unresolved.append((os.path.basename(os.path.dirname(os.path.dirname(p))), tag))
            continue
        if tag in RES: continue
        def num(k, cast=float):
            try: return cast(row.get(k))
            except (TypeError, ValueError): return float("nan")
        RES[tag] = dict(tag=tag, model=kind, L2=num("L2"),
                        params=num("params"), n_iters=num("n_iters"), wall_s=num("wall_s"))
print(f"result rows: {len(RES)}   unresolved: {len(res_unresolved)}")
for a, b in res_unresolved[:10]: print(f"   [unresolved] {a}: {b}")

for p in sorted(INV["generalization"]):
    rel = p.replace(MYDRIVE, "").lstrip("/")
    try: d = np.load(p, allow_pickle=True)
    except Exception as e:
        print(f"   [skip] {rel}: {type(e).__name__}"); continue
    GEN.append(dict(path=rel, keys=list(d.files),
                    shapes={k: list(np.asarray(d[k]).shape) for k in d.files}))
print(f"\ngeneralization files: {len(GEN)}")
for g in GEN: print(f"   {g['path']}\n      keys={g['keys']} shapes={g['shapes']}")

## 8. Figures

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9.5, "axes.titlesize": 10.5,
    "axes.labelsize": 9.5, "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "legend.fontsize": 8.2, "legend.frameon": True, "legend.framealpha": 0.92,
    "legend.edgecolor": "0.8", "axes.grid": True, "grid.alpha": 0.20,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 120, "savefig.dpi": 300})
C_Q, C_T, C_G = "#14285a", "#c0392b", "#2e8b57"
STYLE = {"qapinn": (C_Q, "QAPINN", "o"), "twin": (C_T, "Classical twin", "s"),
         "gaaf": (C_G, "GAAF-PINN", "^")}
def savefig(fig, name):
    p = os.path.join(OUT, name + ".png")
    fig.savefig(p, bbox_inches="tight"); plt.close(fig); print("[fig]", p)

NU = "0.05"
LIVE = {t: v for t, v in HIST.items() if not v["dead"]}

def runs(nu=NU, pde=None, model=None, n=None, families=("MS", "V5", "T1", "CT")):
    out = []
    for t, v in LIVE.items():
        m = v["meta"]
        if m["nu"] != nu: continue
        if m["family"] not in families: continue
        if pde and m["pde"] != pde: continue
        if model and m["model"] != model: continue
        if n is not None and m["n_feat"] != n: continue
        out.append(v)
    return out

# ---- B1: gradient variance, deduplicated, with fits -----------------------
FITS = {}
for nu in ["0.05", "0.01/pi"]:
    rows = [g for g in GRAD if g["nu"] == nu]
    if not rows: continue
    fig, ax = plt.subplots(figsize=(7.4, 4.6))
    for model in ["qapinn", "twin", "gaaf"]:
        pts = [g for g in rows if g["model"] == model]
        if not pts: continue
        col, lab, mk = STYLE[model]
        byser = defaultdict(list)
        for g in pts: byser[g["run"] + "|" + g["series"]].append(g)
        for j, (sk, rr) in enumerate(sorted(byser.items())):
            rr = sorted(rr, key=lambda z: z["n_feat"])
            ax.semilogy([r["n_feat"] for r in rr], [r["variance"] for r in rr],
                        color=col, marker=mk, ms=5, lw=1.6, alpha=0.85,
                        label=lab if j == 0 else None)
            x = np.array([r["n_feat"] for r in rr], float)
            y = np.array([r["variance"] for r in rr], float)
            if len(x) >= 4 and np.all(y > 0):
                b, a = np.polyfit(x, np.log(y), 1)
                r2 = 1 - np.sum((np.log(y)-(a+b*x))**2)/np.sum((np.log(y)-np.log(y).mean())**2)
                FITS[(nu, model, sk)] = (b, r2, len(x), y[0]/y[-1])
                ax.semilogy(x, np.exp(a+b*x), color=col, ls="--", lw=1.0, alpha=0.55)
    ax.set_xlabel("qubit count / feature width $n$")
    ax.set_ylabel("variance of first-layer gradients")
    ax.set_title(f"WS-B: gradient variance against width  ($\\nu={nu}$)", loc="left")
    ax.text(0.02, 0.03, "one line per unique stored series; dashed = exponential fit",
            transform=ax.transAxes, fontsize=7.8, color="0.35")
    ax.legend(loc="best")
    savefig(fig, f"B1_gradient_variance_nu{'005' if nu=='0.05' else 'hard'}")

print("\nexponential fits  var ~ exp(b n)   [series with >=4 points]")
for (nu, model, sk), (b, r2, k, dec) in sorted(FITS.items()):
    print(f"  nu={nu:8s} {model:7s} {sk[:44]:44s} b={b:+.4f}/qubit  "
          f"R2={r2:.3f}  n={k}  total decay {dec:.1f}x")

# ---- B2: loss curves, 3 across and as many rows as needed ----------------
for pde in sorted({v["meta"]["pde"] for v in runs()}):
    ns = sorted({v["meta"]["n_feat"] for v in runs(pde=pde) if v["meta"]["n_feat"]})
    if not ns: continue
    ncol = 3
    nrow = math.ceil(len(ns)/ncol)
    fig, axs = plt.subplots(nrow, ncol, figsize=(4.0*ncol, 3.1*nrow),
                            sharey=True, sharex=True, squeeze=False)
    for idx, n in enumerate(ns):
        ax = axs[idx//ncol][idx % ncol]
        for model in ["qapinn", "twin", "gaaf"]:
            rr = runs(pde=pde, model=model, n=n)
            if not rr: continue
            col, lab, _ = STYLE[model]
            for r in rr:
                ax.semilogy(np.arange(len(r["loss"])), np.maximum(r["loss"], 1e-12),
                            color=col, alpha=0.55, lw=0.9)
            ax.plot([], [], color=col, label=f"{lab} ({len(rr)})")
        # mark where the optimiser changed phase, using the first run that records it
        ref = next((r for r in runs(pde=pde, n=n) if len(r["phase"])), None)
        if ref is not None and len(ref["phase"]):
            ch = np.where(np.diff(ref["phase"]) != 0)[0]
            for c in ch: ax.axvline(c, color="0.6", ls=":", lw=0.8)
        ax.set_title(f"$n={n}$", loc="left")
        ax.legend(loc="upper right", fontsize=7.4)
    for idx in range(len(ns), nrow*ncol): axs[idx//ncol][idx % ncol].axis("off")
    for j in range(ncol): axs[nrow-1][j].set_xlabel("recorded step")
    for i in range(nrow): axs[i][0].set_ylabel("training loss")
    fig.suptitle(f"WS-B: optimisation traces, {pde} at $\\nu=0.05$  "
                 f"(one line per seed; dotted lines mark optimiser phase changes)", y=1.005)
    plt.tight_layout(); savefig(fig, f"B2_loss_curves_{pde}")

# ---- B3: steps to threshold ----------------------------------------------
THRESH = 1e-2
fig, ax = plt.subplots(figsize=(7.0, 4.2))
any3 = False
for model in ["qapinn", "twin", "gaaf"]:
    pts = defaultdict(list)
    for v in runs(model=model):
        n = v["meta"]["n_feat"]
        if n is None: continue
        y = v["loss"]
        pts[n].append(int(np.argmax(y < THRESH)) if np.any(y < THRESH) else np.nan)
    if not pts: continue
    any3 = True
    col, lab, mk = STYLE[model]
    ns = sorted(pts)
    med = [np.nanmedian(pts[n]) if np.any(~np.isnan(pts[n])) else np.nan for n in ns]
    ax.plot(ns, med, color=col, marker=mk, ms=5, lw=1.6, label=lab)
    for n in ns: ax.scatter([n]*len(pts[n]), pts[n], color=col, s=12, alpha=0.35, linewidths=0)
if any3:
    ax.set_xlabel("width $n$"); ax.set_ylabel(f"recorded steps to loss < {THRESH:g}")
    ax.set_title("WS-B: convergence speed against width (median over seeds; "
                 "missing points never reached the threshold)", loc="left")
    ax.legend(loc="best"); savefig(fig, "B3_steps_to_threshold")
else: plt.close(fig)

# ---- B4: residual vs IC balance ------------------------------------------
fig, ax = plt.subplots(figsize=(7.0, 4.2))
any4 = False
for model in ["qapinn", "twin", "gaaf"]:
    xs, ys = [], []
    for v in runs(model=model):
        n = v["meta"]["n_feat"]
        if n is None or not len(v["mse_u"]) or not len(v["mse_f"]): continue
        xs.append(n); ys.append(v["mse_f"][-1]/max(v["mse_u"][-1], 1e-30))
    if not xs: continue
    any4 = True
    col, lab, mk = STYLE[model]
    ns = sorted(set(xs))
    med = [np.median([y for x, y in zip(xs, ys) if x == n]) for n in ns]
    ax.semilogy(ns, med, color=col, marker=mk, ms=5, lw=1.6, label=lab)
    ax.scatter(xs, ys, color=col, s=12, alpha=0.35, linewidths=0)
if any4:
    ax.axhline(1.0, color="k", ls=":", lw=1.0)
    ax.set_xlabel("width $n$"); ax.set_ylabel(r"final $\mathrm{MSE}_f/\mathrm{MSE}_u$")
    ax.set_title("WS-B: which term dominates the converged loss?", loc="left")
    ax.legend(loc="best"); savefig(fig, "B4_residual_vs_ic_balance")
else: plt.close(fig)

# ---- B5: accuracy and cost. Cost now comes from the histories -------------
fig, axs = plt.subplots(1, 2, figsize=(11.0, 4.0))
okA = okB = False
for model in ["qapinn", "twin", "gaaf"]:
    rr = [v for v in runs(model=model) if v["meta"]["n_feat"]]
    if not rr: continue
    col, lab, mk = STYLE[model]
    # (a) accuracy: L2 from master_results where available, else final loss
    pa = [(v["meta"]["n_feat"],
           RES.get(v["tag"], {}).get("L2", float("nan")) if np.isfinite(
               RES.get(v["tag"], {}).get("L2", float("nan"))) else v["loss"][-1]) for v in rr]
    pa = [(n, y) for n, y in pa if np.isfinite(y)]
    if pa:
        okA = True
        ns = sorted({n for n, _ in pa})
        axs[0].semilogy(ns, [np.median([y for n2, y in pa if n2 == n]) for n in ns],
                        color=col, marker=mk, ms=5, lw=1.6, label=lab)
        axs[0].scatter([n for n, _ in pa], [y for _, y in pa],
                       color=col, s=12, alpha=0.35, linewidths=0)
    # (b) cost from the checkpoint history: wall_s / recorded steps
    pb = [(v["meta"]["n_feat"], v["wall_s"]/max(len(v["loss"]), 1)) for v in rr
          if np.isfinite(v["wall_s"]) and v["wall_s"] > 0]
    if pb:
        okB = True
        ns = sorted({n for n, _ in pb})
        axs[1].semilogy(ns, [np.median([y for n2, y in pb if n2 == n]) for n in ns],
                        color=col, marker=mk, ms=5, lw=1.6, label=lab)
        axs[1].scatter([n for n, _ in pb], [y for _, y in pb],
                       color=col, s=12, alpha=0.35, linewidths=0)
axs[0].set_xlabel("width $n$"); axs[0].set_ylabel(r"relative $L^2$ (or final loss)")
axs[0].set_title("(a) accuracy against width", loc="left")
axs[1].set_xlabel("width $n$"); axs[1].set_ylabel("wall seconds per recorded step")
axs[1].set_title("(b) cost against width", loc="left")
if okA: axs[0].legend(loc="best")
if okB: axs[1].legend(loc="best")
else:
    axs[1].text(0.5, 0.5, "no wall-clock data in the stored histories",
                ha="center", va="center", transform=axs[1].transAxes, color="0.4")
    print("[warn] B5(b): no run recorded wall_s; panel left explicitly empty")
fig.text(0.5, -0.04, "Wall-clock is not comparable across machines; panel (b) shows relative "
         "scaling within each family only.", ha="center", fontsize=7.8, color="0.35")
plt.tight_layout(); savefig(fig, "B5_accuracy_and_cost")

## 9. Write CSVs and manifest

In [ ]:
srows = []
for t, v in HIST.items():
    m = v["meta"]; y = v["loss"]; r = RES.get(t, {})
    srows.append(dict(tag=t, path=v["path"], family=m["family"], nu=m["nu"], pde=m["pde"],
                      model=m["model"], n_feat=m["n_feat"], seed=m["seed"], retry=m["retry"],
                      note=m["note"], dead=v["dead"], n_steps=len(y),
                      loss_first=float(y[0]), loss_final=float(y[-1]), loss_min=float(y.min()),
                      mse_u_final=float(v["mse_u"][-1]) if len(v["mse_u"]) else float("nan"),
                      mse_f_final=float(v["mse_f"][-1]) if len(v["mse_f"]) else float("nan"),
                      wall_s=v["wall_s"],
                      L2=r.get("L2", float("nan")), params=r.get("params", float("nan"))))
def dump(rows, name):
    if not rows: return
    p = os.path.join(OUT, name)
    with open(p, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print("[saved]", p, f"({len(rows)} rows)")

dump(srows, "wsb_run_summary.csv")
dump(GRAD, "wsb_gradient_variance.csv")
dump(ACCOUNT, "wsb_checkpoint_accounting.csv")

lrows = []
for t, v in HIST.items():
    y = v["loss"]; step = max(1, len(y)//400)
    for i in range(0, len(y), step):
        lrows.append(dict(tag=t, nu=v["meta"]["nu"], pde=v["meta"]["pde"],
                          model=v["meta"]["model"], n_feat=v["meta"]["n_feat"],
                          seed=v["meta"]["seed"], step=i, loss=float(y[i]),
                          mse_u=float(v["mse_u"][i]) if i < len(v["mse_u"]) else float("nan"),
                          mse_f=float(v["mse_f"][i]) if i < len(v["mse_f"]) else float("nan"),
                          phase=float(v["phase"][i]) if i < len(v["phase"]) else float("nan")))
dump(lrows, "wsb_loss_traces.csv")

man = dict(
    note="wsb-learning-dynamics (Track A, Djabon) - revision 2",
    inventory={k: len(v) for k, v in INV.items()},
    accounting=dict(checkpoints_found=N_CKPT, kept=len(kept), excluded=len(excl),
                    exclusions_by_reason=dict(Counter(a["reason"] for a in excl))),
    dead_runs=dead_runs,
    gradient=dict(series_found=len(SERIES), duplicates_dropped=len(dupes),
                  duplicate_pairs=[{"dropped": f"{a[0]} :: {a[1]}",
                                    "identical_to": f"{b[0]} :: {b[1]}"}
                                   for a, b in dupes.items()],
                  unique_rows=len(GRAD),
                  unlabelled_series_attributed_to_qapinn=[
                      f"{S['path']} :: {S['series']}" for k, S in SERIES.items()
                      if not S["labelled"] and k not in dupes],
                  missing=[f"nu={nu} model={m}" for nu, m in missing]),
    fits={f"{k[0]}|{k[1]}|{k[2]}": dict(slope_per_qubit=v[0], r2_log=v[1],
                                        n_points=v[2], total_decay=v[3])
          for k, v in FITS.items()},
    n_results=len(RES), n_generalization=len(GEN),
    coverage={f"{k[0]}|{k[1]}|{k[2]}|n{k[3]}|{k[4]}": sorted(v) for k, v in cov.items()},
    caveats=[
        "nu=0.05 and nu=0.01/pi are catalogued separately and never merged.",
        "Exact-duplicate gradient series are excluded from fits; the pairs are listed above.",
        "Gradient files record a qubit axis and variance series but NOT the PDE; the PDE is "
        "inferred from the run folder shown in the path.",
        "Series named 'var' carry no model label and are attributed to QAPINN because the "
        "notebooks that wrote them swept QAPINN only.",
        "Loss histories record one point per Adam iteration and one per L-BFGS chunk, so the "
        "step axis is not a uniform iteration count; phase changes are marked in B2.",
        "Wall-clock is not comparable across machines.",
        "Dead runs are detected from their own histories and excluded from figures, not from "
        "the CSVs.",
        "No value is interpolated or filled; missing configurations are reported as missing.",
    ])
p = os.path.join(OUT, "wsb_manifest.json")
json.dump(man, open(p, "w"), indent=2)
print("\n[saved]", p)
print(json.dumps({k: man[k] for k in ("inventory", "accounting", "gradient")}, indent=2)[:2200])

## 10. Archive the notebook with the results

In [ ]:
NB = "WSB_LearningDynamics_Colab.ipynb"
cands = [q for q in glob.glob(os.path.join(MYDRIVE, "**", "*.ipynb"), recursive=True)
         if os.path.basename(q).startswith("WSB_LearningDynamics_Colab")]
if cands:
    src = max(cands, key=os.path.getmtime); dst = os.path.join(OUT, NB)
    if os.path.abspath(src) != os.path.abspath(dst): shutil.copy2(src, dst)
    print("[saved] notebook archived to", dst)
else:
    print("[note] notebook not found in Drive; use File > Save a copy in Drive and re-run.")

print("\nFinal contents of", OUT)
for f in sorted(os.listdir(OUT)):
    print(f"   {f:42s} {os.path.getsize(os.path.join(OUT, f)):>12,d} bytes")